# 01 — Data Cleaning
**Project:** AI-Based Real Estate Plot Scheme Prediction
**Purpose:** Load the raw merged listings dataset, clean it, handle missing values, fix data types, and save a processed version for EDA and modeling.

**Input:** `data/raw/real_estate_master.csv`
**Output:** `data/processed/cleaned_listings.csv`


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

RAW_PATH = "../data/raw/real_estate_master.csv"
OUT_PATH = "../data/processed/cleaned_listings.csv"

df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()


Shape: (13828, 41)


,location,area,price,price_currency,status,new/resale,price_negotiable,description,security_deposit,facing,furnished,age of property,Lift(s),Full Power Backup,24 X 7 Security,Children's play area,Club House,Gymnasium,Swimming Pool,Sports Facility,Jogging Track,Landscaped Gardens,locality_score,project_score,builder_experience,Intercom,Indoor Games,ATM,Maintenance Staff,Staff Quarter,Multipurpose Room,Car Parking,Hospital,School,Shopping Mall,Vaastu Compliant,Cafeteria,Rain Water Harvesting,Golf Course,city,property_type
0,Dhakoli,1300.0,2850000.0,INR,NaN,0.0,0.0,This spacious 2 bhk builder floor is available...,1.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chandigarh,builderfloor
1,Dhakoli,1400.0,3600000.0,INR,NaN,0.0,0.0,It’s a 3 bhk builder floor situated in Dhakoli...,1.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chandigarh,builderfloor
2,Dhakoli,1350.0,3690000.0,INR,NaN,0.0,0.0,It has an area of 1350 sqft with a carpet area...,1.0,northeast,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chandigarh,builderfloor
3,New chandigarh mohali,1200.0,10000000.0,INR,NaN,0.0,0.0,It has a salable area of 1200 sqft and is avai...,1.0,northeast,1.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chandigarh,builderfloor
4,Sunny Enclave,1008.0,3090000.0,INR,NaN,1.0,0.0,This spacious 2 bhk builder floor is available...,1.0,east,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chandigarh,builderfloor


## 1. Initial Inspection

In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13828 entries, 0 to 13827
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   location               13814 non-null  object 
 1   area                   13814 non-null  float64
 2   price                  13815 non-null  float64
 3   price_currency         13815 non-null  object 
 4   status                 4690 non-null   float64
 5   new/resale             13814 non-null  float64
 6   price_negotiable       13814 non-null  float64
 7   description            13264 non-null  object 
 8   security_deposit       13814 non-null  float64
 9   facing                 11807 non-null  object 
 10  furnished              13814 non-null  float64
 11  age of property        13828 non-null  int64  
 12  Lift(s)                13814 non-null  float64
 13  Full Power Backup      13814 non-null  float64
 14  24 X 7 Security        13814 non-null  float64
 15  Ch

In [3]:
null_pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
null_pct.head(20)


project_score            96.43
Golf Course              95.26
Cafeteria                92.79
builder_experience       91.01
Multipurpose Room        90.71
Indoor Games             89.88
Staff Quarter            89.22
Maintenance Staff        87.12
Rain Water Harvesting    86.06
Shopping Mall            85.99
ATM                      85.99
Hospital                 84.83
Vaastu Compliant         83.52
School                   81.64
Intercom                 79.52
Car Parking              71.93
locality_score           68.93
status                   66.08
facing                   14.62
description               4.08
dtype: float64

## 2. Drop Columns That Are Mostly Empty
Columns with >75% missing values carry little signal and can't be reliably imputed.
We drop these rather than fabricate values for three-quarters of the dataset.


In [4]:
drop_threshold = 0.75
cols_to_drop = null_pct[null_pct > drop_threshold * 100].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns (>{int(drop_threshold*100)}% null):")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)
print("\nNew shape:", df.shape)


Dropping 15 columns (>75% null):
['project_score', 'Golf Course', 'Cafeteria', 'builder_experience', 'Multipurpose Room', 'Indoor Games', 'Staff Quarter', 'Maintenance Staff', 'Rain Water Harvesting', 'Shopping Mall', 'ATM', 'Hospital', 'Vaastu Compliant', 'School', 'Intercom']

New shape: (13828, 26)


## 3. Fix Data Types

In [5]:
# price and area should be numeric
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["area"] = pd.to_numeric(df["area"], errors="coerce")
df["security_deposit"] = pd.to_numeric(df["security_deposit"], errors="coerce")

# drop rows where price or area could not be parsed / are missing
before = len(df)
df = df.dropna(subset=["price", "area"])
after = len(df)
print(f"Dropped {before - after} rows with unparseable price/area")


Dropped 14 rows with unparseable price/area


## 4. Remove Invalid / Impossible Values

In [6]:
# Remove non-positive or extreme outlier price/area values (data entry errors)
before = len(df)
df = df[(df["price"] > 0) & (df["area"] > 0)]

# Cap extreme outliers using the 1st and 99th percentile per property_type
def filter_outliers_by_group(data, col, group_col):
    lower = data.groupby(group_col)[col].transform(lambda s: s.quantile(0.01))
    upper = data.groupby(group_col)[col].transform(lambda s: s.quantile(0.99))
    return data[(data[col] >= lower) & (data[col] <= upper)]

df = filter_outliers_by_group(df, "price", "property_type")
df = filter_outliers_by_group(df, "area", "property_type")

after = len(df)
print(f"Removed {before - after} outlier/invalid rows. New shape: {df.shape}")


Removed 509 outlier/invalid rows. New shape: (13305, 26)


## 5. Standardize Categorical / Binary Columns

In [7]:
# Identify amenity flag columns (originally 0/1 but may load as float due to NaNs)
amenity_cols = [
    "Lift(s)", "Full Power Backup", "24 X 7 Security", "Children's play area",
    "Club House", "Gymnasium", "Swimming Pool", "Sports Facility",
    "Jogging Track", "Landscaped Gardens"
]
amenity_cols = [c for c in amenity_cols if c in df.columns]

for c in amenity_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Standardize text categoricals
for c in ["facing", "status", "location", "city", "property_type"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.title()
        df[c] = df[c].replace({"Nan": np.nan})

df[amenity_cols + ["facing", "status"]].head()


,Lift(s),Full Power Backup,24 X 7 Security,Children's play area,Club House,Gymnasium,Swimming Pool,Sports Facility,Jogging Track,Landscaped Gardens,facing,status
0,0,0,0,0,0,0,0,0,0,0,NaN,NaN
1,0,0,0,0,0,0,0,0,0,0,NaN,NaN
2,0,0,0,0,0,0,0,0,0,0,Northeast,NaN
3,0,0,0,0,0,0,0,0,0,0,Northeast,NaN
4,0,0,0,0,0,0,0,0,0,0,East,NaN


## 6. Handle Remaining Missing Values

In [8]:
# facing / status: keep as an explicit "Unknown" category rather than dropping rows
df["facing"] = df["facing"].fillna("Unknown")
df["status"] = df["status"].fillna("Unknown")

# description: fill blank with empty string (not used numerically)
if "description" in df.columns:
    df["description"] = df["description"].fillna("")

# Any remaining nulls?
remaining_nulls = df.isna().sum()
remaining_nulls[remaining_nulls > 0]


locality_score    9171
Car Parking       9577
dtype: int64

## 7. Feature Derivations Useful Downstream

In [9]:
df["price_per_sqft"] = (df["price"] / df["area"]).round(2)

# Total amenity count as a simple composite feature
df["amenity_count"] = df[amenity_cols].sum(axis=1)

df[["price", "area", "price_per_sqft", "amenity_count"]].describe()


,price,area,price_per_sqft,amenity_count
count,1.330500e+04,13305.000000,13305.000000,13305.000000
mean,6.545499e+06,1448.644946,3920.916552,2.773393
std,8.607935e+06,822.723808,2916.970498,2.844853
min,2.650000e+05,353.000000,100.000000,0.000000
25%,1.575000e+06,1000.000000,1460.670000,0.000000
50%,4.258000e+06,1200.000000,3720.000000,2.000000
75%,7.500000e+06,1650.000000,5294.120000,5.000000
max,1.150000e+08,9950.000000,41666.670000,10.000000


## 8. Remove Duplicate Rows

In [10]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} exact duplicate rows")


Removed 946 exact duplicate rows


## 9. Final Check & Save

In [11]:
print("Final shape:", df.shape)
print("\nRows per city:")
print(df["city"].value_counts())
print("\nRows per property_type:")
print(df["property_type"].value_counts())

df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")


Final shape: (12359, 28)

Rows per city:
city
Ghaziabad     6006
Pune          2511
Lucknow       2007
Chandigarh    1835
Name: count, dtype: int64

Rows per property_type:
property_type
Builderfloor    6221
Plot            4656
Villa           1482
Name: count, dtype: int64

Saved cleaned dataset to ../data/processed/cleaned_listings.csv


---
**Next notebook:** `02_eda_visualization.ipynb` — explore distributions, correlations, and price drivers on this cleaned dataset.
